# Q3 · Steps 7–9 — API test and deployment package

This notebook (1) tests the FastAPI service **in-process**, and (2) builds the folder `Q3/deploy_hf/` that you upload to **Hugging Face Spaces** (Docker Space running FastAPI + Streamlit together).

### How to run the API and UI locally (two terminals, project root, environment active)
```
uvicorn main:app --port 8000 --app-dir Q3/api          # terminal 1 -> http://127.0.0.1:8000/docs
streamlit run Q3/app/streamlit_app.py                  # terminal 2 -> http://localhost:8501
```

In [1]:
# ---- Project setup: finds the project root automatically ----
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def find_root():
    start = Path(os.environ.get("LAB_ROOT", Path.cwd())).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Project root not found. Open this notebook from inside the Prog_for_AI_Lab folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

Project root: C:\Users\hp\Desktop\Prog_for_AI_Lab\Prog_for_AI_Lab


In [2]:
import shutil, platform, importlib.metadata as md
sys.path.insert(0, str(ROOT / "Q3" / "api"))
from fastapi.testclient import TestClient
import main as api_module

client = TestClient(api_module.app)
print(client.get("/health").json())
print(client.get("/metrics").json())
print(client.get("/feature-importance").json())
sample = {"age": 39, "workclass": "Private", "education_num": 13, "marital_status": "Married-civ-spouse", "occupation": "Prof-specialty",
          "relationship": "Husband", "race": "White", "sex": "Male", "capital_gain": 0, "capital_loss": 0, "hours_per_week": 45, "native_country": "United-States"}
r = client.post("/predict", json=sample); print(r.status_code, r.json())
assert r.status_code == 200 and "prediction" in r.json()
bad = client.post("/predict", json={**sample, "age": 5}); print("invalid age ->", bad.status_code, "(expected 422)")

{'status': 'ok', 'model': 'Baseline: Logistic Regression'}
{'model_name': 'Baseline: Logistic Regression', 'test_metrics': {'accuracy': 0.8569015814524796, 'precision': 0.7438650306748467, 'recall': 0.6186224489795918, 'f1': 0.6754874651810585, 'roc_auc': 0.908843644889705}}
{'method': 'permutation importance (drop in ROC-AUC)', 'top_features': [{'feature': 'marital_status', 'importance': 0.09603}, {'feature': 'capital_gain', 'importance': 0.03783}, {'feature': 'age', 'importance': 0.03261}, {'feature': 'education_num', 'importance': 0.02988}, {'feature': 'occupation', 'importance': 0.01553}]}
200 {'prediction': 1, 'label': '>50K', 'probability_over_50k': 0.7533}
invalid age -> 422 (expected 422)


## Build the Hugging Face Spaces package (`Q3/deploy_hf/`)

In [3]:
D = ROOT / "Q3" / "deploy_hf"
def w(path, text):
    # UTF-8 + Linux line endings (needed for Docker/Hugging Face even when running on Windows)
    path.write_text(text, encoding="utf-8", newline="\n")
if D.exists(): shutil.rmtree(D)
(D / "models").mkdir(parents=True)
shutil.copy(ROOT / "Q3" / "api" / "main.py", D / "main.py")
shutil.copy(ROOT / "Q3" / "app" / "streamlit_app.py", D / "streamlit_app.py")
shutil.copy(ROOT / "src" / "q3_common.py", D / "q3_common.py")
shutil.copy(ROOT / "models" / "best_model.joblib", D / "models" / "best_model.joblib")
shutil.copy(ROOT / "models" / "model_metadata.json", D / "models" / "model_metadata.json")

# pin the SAME library versions that created the pickle (a pickle must be loaded with matching versions)
def v(pkg):
    try: return f"{pkg}=={md.version(pkg)}"
    except md.PackageNotFoundError: return None
pkgs = ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "fastapi", "uvicorn", "streamlit", "requests", "pydantic"]
if b"tpot" in (ROOT / "models" / "best_model.joblib").read_bytes(): pkgs.append("tpot")
blob = (ROOT / "models" / "best_model.joblib").read_bytes()
extra = [p for p in ["xgboost", "lightgbm"] if p.encode() in blob]
req = list(filter(None, map(v, pkgs + extra)))
if "tpot" in pkgs: req.append("setuptools<81")      # TPOT's dependency 'stopit' needs pkg_resources
w(D / "requirements.txt", "\n".join(req) + "\n")

py = f"{sys.version_info.major}.{sys.version_info.minor}"
w(D / "Dockerfile", f'''FROM python:{py}-slim
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV MODEL_DIR=/app/models API_URL=http://127.0.0.1:8000
EXPOSE 7860
CMD ["bash", "start.sh"]
''')
w(D / "start.sh", "\n".join([
    "#!/bin/bash",
    "uvicorn main:app --host 127.0.0.1 --port 8000 &",
    "sleep 6",
    "streamlit run streamlit_app.py --server.port 7860 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --server.headless true",
]) + "\n")
w(D / "README.md", '''---
title: Adult Income Predictor
emoji: 💰
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---
# Adult Income Predictor
FastAPI (model service) + Streamlit (user interface) – Programming for AI mid-term lab, Question 3.
''')
print("Deployment package ready:", D)
for p in sorted(D.rglob("*")): print(" ", p.relative_to(D))
print("\nrequirements.txt:\n" + (D / "requirements.txt").read_text(encoding="utf-8"))

Deployment package ready: C:\Users\hp\Desktop\Prog_for_AI_Lab\Prog_for_AI_Lab\Q3\deploy_hf
  Dockerfile
  main.py
  models
  models\best_model.joblib
  models\model_metadata.json
  q3_common.py
  README.md
  requirements.txt
  start.sh
  streamlit_app.py

requirements.txt:
numpy==2.4.6
pandas==3.1.1
scipy==1.17.1
scikit-learn==1.9.1
joblib==1.6.0
fastapi==0.165.0
uvicorn==0.54.0
streamlit==1.67.0
requests==2.34.2
pydantic==2.14.3



## Manual steps to deploy (do these yourself)
1. Create a free account on **huggingface.co** → *New Space* → name e.g. `adult-income-predictor` → SDK **Docker** → *Blank* → Public.
2. In the Space click **Files → Add file → Upload files** and upload **everything inside `Q3/deploy_hf/`** (including the `models` folder, `Dockerfile`, `start.sh`, `README.md`).
   (Or `git clone` the Space and copy the files in, then `git add . && git commit -m "deploy" && git push`.)
3. Wait for the build (3–10 min, watch the *Logs* tab). When it says *Running*, open the app.
4. Copy the Space URL (`https://huggingface.co/spaces/<username>/adult-income-predictor`) into `reports/report_config.json` → `"live_url"`.
5. Test the live app with 2–3 different inputs and take a screenshot for the report (`reports/screenshots/`).

**Alternative:** Streamlit Community Cloud needs the FastAPI service to be hosted separately, so Hugging Face Docker is simpler.